# 🎯 HMM Market Regime Detection
## نماذج ماركوف المخفية لاكتشاف أنظمة السوق

**المرجع النظري:** Stanford HMM Framework (CS229 / CS228)  
**التطبيق:** Market Regime Detection in Quantitative Trading  
**المكتبة:** hmmlearn — GaussianHMM  

---

### الإطار النظري (Stanford HMM)

وفقاً لمواد Stanford CS229/CS228، يُعرَّف HMM رياضياً بـ:
- **S** = مجموعة الحالات المخفية (أنظمة السوق)
- **O** = متتالية الملاحظات (خصائص السوق اليومية)
- **π** = توزيع الحالة الابتدائية
- **A** = مصفوفة انتقال الأنظمة
- **B** = توزيع الانبعاث الغاوسي لكل نظام

**الخوارزميات الأساسية:**
- `Baum-Welch` → تعلم معاملات النموذج
- `Viterbi`    → استخراج أفضل تسلسل للأنظمة
- `Forward`    → تقييم احتمالية التسلسل

In [ ]:
import sys, warnings
from pathlib import Path

# Add src to path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('✅ Environment ready')
print(f'Project root: {PROJECT_ROOT}')

## الخطوة 1: البحث عن ملف Stanford HMM

In [ ]:
from file_finder import find_repo_root, print_search_report
from pdf_reader  import extract_pdf_text

root = find_repo_root(PROJECT_ROOT)
stanford_pdf = print_search_report(root)
hmm_text = extract_pdf_text(stanford_pdf)
print(f'\n📚 Theory content loaded: {len(hmm_text):,} characters')

## الخطوة 2: تحميل بيانات السوق

In [ ]:
from data_loader import download_data

df = download_data('^GSPC', start='2010-01-01', end='2024-12-31')
print(df.describe().round(2))

## الخطوة 3: بناء الخصائص (Observations)

In [ ]:
from features import compute_features, get_feature_matrix

X, dates, scaler = get_feature_matrix(df)
feat_df = compute_features(df).loc[dates]

print(f'\nFeature matrix shape: {X.shape}')
print('Features (Observations for HMM):', list(feat_df.columns[:5]))

## الخطوة 4: تدريب HMM (Baum-Welch)

In [ ]:
from hmm_model import train_hmm, decode_states, print_model_summary

model  = train_hmm(X, n_states=3, n_iter=300)
states = decode_states(model, X)
print_model_summary(model, states)

## الخطوة 5: تفسير الأنظمة المكتشفة

In [ ]:
from regime_interpreter import interpret_regimes, build_regime_series

stats_df, labels = interpret_regimes(feat_df, states, df)
regime_df = build_regime_series(dates, states, labels)
print('\nRegime labels:')
for sid, lbl in labels.items():
    print(f'  State {sid}: {lbl}')

## الخطوة 6: الباكتيست التعليمي

In [ ]:
from backtest import build_signal, run_backtest

signal_df = build_signal(regime_df, labels)
results_df, metrics = run_backtest(df, signal_df)

print('\n📊 Key Metrics:')
print(f"  Strategy Total Return: {metrics['Strategy_total_return']*100:.1f}%")
print(f"  Buy&Hold Total Return: {metrics['BuyHold_total_return']*100:.1f}%")
print(f"  Strategy Sharpe:       {metrics['Strategy_sharpe']:.2f}")
print(f"  Strategy Max DD:       {metrics['Strategy_max_drawdown']*100:.1f}%")

## الخطوة 7: الرسوم البيانية

In [ ]:
from visualization import generate_all_charts
from hmm_model import get_model_summary

summary = get_model_summary(model, states)
generate_all_charts(df, regime_df, labels, stats_df, results_df, summary['transition_matrix'])
print('\n✅ Charts saved to outputs/figures/')

## خلاصة تعليمية

### ما تعلمناه:
1. **HMM يكتشف الأنظمة تلقائياً** — بدون تسميات يدوية
2. **Baum-Welch** يتعلم من البيانات عبر EM Algorithm
3. **Viterbi** يعطي أفضل تسلسل للأنظمة التاريخية
4. **الأنظمة قابلة للتفسير** — كل حالة لها إحصاءات مميزة
5. **الاستراتيجية تُقلل المخاطر** (Max Drawdown أقل) على حساب بعض العائد

### تحذير مهم:
> ⚠ هذا الباكتيست داخل العيّنة (In-Sample) ويحتوي على تحيّزات متعددة.  
> لا يُمثّل نصيحة مالية بأي شكل من الأشكال.